# Agent 2 — University Matching Agent (v3)
RAG (Qdrant + BAAI embeddings + CrossEncoder rerank) + RandomForest admit-probability classifier + SHAP explainability + Groq LLaMA explanation.

**v3 fixes:**
1. Fixed concatenated/broken statements from a notebook-generation bug (missing newlines in several cells)
2. Fixed `_sample_shap` vs `sample_shap` variable-name mismatch
3. Fixed `admit_model = grid_search.best_estimator_print(...)` broken line
4. `predict_proba` now looks up the positive-class index via `admit_model.classes_` instead of assuming `[0][1]`
5. Added `CalibratedClassifierCV` — the model is now actually calibrated, not just checked; renamed rest as *model-estimated* admission probability
6. University ranking now uses a combined `overall_score` (match_score + admit_probability), not admit_probability alone
7. Groq explanation prompt now receives the normalized GPA (out of 4.0), not the raw uploaded GPA/scale

Architecture kept as-is per review: retrieve/rerank/score at program level, then GROUP BY university for the shortlist — Agent 3 still owns program-level matching downstream.

## 1. Setup

In [ ]:
!pip install qdrant-client sentence-transformers groq shap sqlalchemy psycopg2-binary scikit-learn -q

In [2]:
from google.colab import files
print("Upload agent2_final_2tier.csv (program/university knowledge base)")
uploaded = files.upload()

Upload agent2_final_2tier.csv (program/university knowledge base)


Saving agent2_final_2tier.csv to agent2_final_2tier.csv


In [3]:
import pandas as pd

df = pd.read_csv("agent2_final_2tier.csv")
df = df.fillna("")
print(df.shape)
print(df.columns.tolist())

# The framework's university-level output needs country / region / institution_type.
# If your CSV doesn't have these columns, add them (or merge from Agent 4's dataset,
# which already has control_code / institution_sector_code for public/private).
for required_col in ["country", "region", "institution_type"]:
    if required_col not in df.columns:
        print(f"WARNING: '{required_col}' column missing — output will show 'Unknown' until added.")

print(df[['university_name','program_name','degree_type','tier','tier_score']].head())

(4500, 44)
['university_id', 'university_name', 'state', 'city', 'latitude', 'longitude', 'tuition_in_state_usd', 'tuition_out_state_usd', 'university_admission_rate', 'student_size', 'us_news_ranking', 'world_ranking', 'campus_setting', 'website_url', 'avg_cost_of_living_monthly', 'international_student_pct', 'public_or_private', 'program_id', 'program_name', 'degree_type', 'department', 'min_gpa', 'min_gre_quant', 'min_gre_verbal', 'min_gmat', 'min_toefl', 'min_ielts', 'application_deadline_fall', 'application_fee_usd', 'duration_months', 'stem_designated', 'program_admit_rate', 'description_text', 'curriculum_highlights', 'faculty_research_areas', 'avg_starting_salary_usd', 'male_acceptance_rate', 'female_acceptance_rate', 'historical_admit_count', 'historical_admit_rate', 'data_quality', 'track', 'tier_score', 'tier']
                         university_name  \
0  Massachusetts Institute of Technology   
1  Massachusetts Institute of Technology   
2  Massachusetts Institute of Tech

## 2. Vector store — Qdrant (programs/universities knowledge base)

In [4]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

# ":memory:" only persists for this Colab session.
# For the real pipeline (frontend/backend connected), point this at Qdrant Cloud instead:
# client = QdrantClient(url="<your-qdrant-cloud-url>", api_key="<your-qdrant-api-key>")
client = QdrantClient(":memory:")

In [5]:
from sentence_transformers import SentenceTransformer

# embed_model (not "model") to avoid namespace collisions when merged with other agents.
embed_model = SentenceTransformer('BAAI/bge-small-en-v1.5')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
def build_full_text(row):
    """Full document text — used for BOTH embedding and reranking, so the
    CrossEncoder judges the same information the retriever indexed on."""
    return (
        f"{row['program_name']} at {row['university_name']}. "
        f"{row['description_text']} "
        f"{row['curriculum_highlights']} "
        f"{row['faculty_research_areas']}"
    )

collection_name = "kb_universities"

if client.collection_exists(collection_name):
    client.delete_collection(collection_name)

client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=384, distance=Distance.COSINE)
)

texts = [build_full_text(row) for _, row in df.iterrows()]
vectors = embed_model.encode(texts, batch_size=32, show_progress_bar=True)

points = []
for i, (_, row) in enumerate(df.iterrows()):
    payload = row.to_dict()
    payload["full_text"] = texts[i]   # store so reranker can reuse it later
    points.append(PointStruct(id=i, vector=vectors[i].tolist(), payload=payload))

client.upsert(collection_name=collection_name, points=points)
print(f"Successfully uploaded {len(points)} programs")

Batches:   0%|          | 0/141 [00:00<?, ?it/s]

Successfully uploaded 4500 programs


## 3. API keys

In [7]:
from google.colab import userdata
import os
os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')

try:
    POSTGRES_URL = userdata.get("POSTGRES_URL")
except Exception:
    POSTGRES_URL = None

## 4. Retrieval + reranking
Budget is a **soft** signal only — Agent 4 (Financial) owns real affordability. Agent 2 just tags each candidate with a rough `within_stated_budget` flag.

In [8]:
from qdrant_client.models import Filter, FieldCondition, MatchValue, Range

def retrieve_candidates(profile, preferences, top_k=20):
    query_text = f"{profile['target_degree']} {profile['target_program']}"
    query_vector = embed_model.encode(query_text).tolist()

    # Only hard-filter on degree_type (a real eligibility constraint).
    # Budget is NOT a hard filter — affordability belongs to Agent 4.
    must = [FieldCondition(key="degree_type", match=MatchValue(value=profile["target_degree"]))]

    results = client.query_points(
        collection_name="kb_universities",
        query=query_vector,
        query_filter=Filter(must=must),
        limit=top_k
    )
    return results.points


def tag_budget_signal(payload, preferences):
    """Informational only — not used to exclude candidates."""
    budget_max = preferences.get("budget_max_usd")
    tuition = payload.get("tuition_out_state_usd")
    if not budget_max or not tuition:
        return None
    try:
        return float(tuition) <= float(budget_max) * 1.15
    except (ValueError, TypeError):
        return None

In [9]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder('BAAI/bge-reranker-base')


def rerank(query_text, candidates, top_n=10):
    """Reranker sees the FULL document (program + university + description +
    curriculum + faculty research), matching what was embedded — not just description_text."""
    if not candidates:
        return []
    pairs = [(query_text, c.payload.get("full_text", c.payload.get("description_text", ""))) for c in candidates]
    scores = reranker.predict(pairs)
    return sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)[:top_n]

config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

## 5. Scoring helpers
`priority` (`ranking` / `research` / `cost` / `location`) reweights the scoring formula instead of being ignored.

In [10]:
def normalize_gpa_to_4(gpa, gpa_scale):
    """Change any scale to 4.0 scale GPA."""
    try:
        gpa = float(gpa)
        gpa_scale = float(gpa_scale) if gpa_scale else 4.0
        if gpa_scale <= 0:
            return gpa
        return (gpa / gpa_scale) * 4.0
    except (ValueError, TypeError):
        return None


# Priority -> weight profile for (semantic, tier, gpa_fit, extracurricular)
PRIORITY_WEIGHTS = {
    "ranking":  (0.30, 0.35, 0.20, 0.15),   # emphasize program tier/prestige
    "research": (0.40, 0.20, 0.20, 0.20),   # emphasize semantic fit to research areas
    "cost":     (0.30, 0.20, 0.30, 0.20),   # de-emphasize tier — cost is Agent 4's real job
    "location": (0.30, 0.25, 0.25, 0.20),
    "default":  (0.35, 0.25, 0.25, 0.15),
}

def compute_final_score(payload, profile, semantic_score, extracurricular_score=0.5, priority="default"):
    w_sem, w_tier, w_gpa, w_extra = PRIORITY_WEIGHTS.get(priority, PRIORITY_WEIGHTS["default"])

    program_tier_score = float(payload.get("tier_score", 50)) / 100
    normalized_semantic = 1 / (1 + pow(2.718, -semantic_score))

    try:
        student_gpa = normalize_gpa_to_4(profile.get("gpa", 3.0), profile.get("gpa_scale", 4.0))
        min_gpa = float(payload.get("min_gpa", 3.0) or 3.0)
        if student_gpa is None:
            raise ValueError
        gpa_fit = max(0, min(1, 1 - abs(student_gpa - min_gpa) / 4.0))
    except (ValueError, TypeError):
        gpa_fit = 0.5

    final_score = (
        w_sem * normalized_semantic
        + w_tier * program_tier_score
        + w_gpa * gpa_fit
        + w_extra * extracurricular_score
    )
    return round(final_score, 3)

## 6. Admit-probability model — RandomForestClassifier
Trained on real merged data (historical admits + program tiers). `extracurricular_score` is a 6th feature so Agent 7's output improves this model once it's producing real scores — defaults to 0.5 (neutral) until then.

In [11]:
print("Upload us_historical_admits_no_phd.csv")
uploaded = files.upload()

Upload us_historical_admits_no_phd.csv


Saving us_historical_admits_no_phd.csv to us_historical_admits_no_phd.csv


In [12]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np

admits = pd.read_csv("us_historical_admits_no_phd.csv")
programs = pd.read_csv("agent2_final_2tier.csv")

print("admits.csv columns:", admits.columns.tolist())
print("programs.csv columns:", programs.columns.tolist())
# Inspect the printout above — if admits.csv has more columns than gpa/gre bands
# (e.g. undergrad institution tier, work experience, major, SOP flag), add them below.
# Weak accuracy is very often a feature problem, not a model problem.

def band_to_midpoint(band_str):
    """'3.4-3.5' -> 3.45, missing -> None"""
    if pd.isna(band_str) or band_str == "":
        return None
    try:
        low, high = band_str.split("-")
        return (float(low) + float(high)) / 2
    except:
        return None

admits["gpa_numeric"] = admits["student_gpa_band"].apply(band_to_midpoint)
admits["gre_numeric"] = admits["student_gre_band"].apply(band_to_midpoint)

prog_subset = programs[["program_id", "tier_score", "program_admit_rate", "min_gpa", "min_gre_quant"]]
prog_subset = prog_subset.drop_duplicates(subset="program_id")
merged = admits.merge(prog_subset, on="program_id", how="left")

merged["gre_missing"] = merged["gre_numeric"].isna().astype(int)
merged["gpa_numeric"] = merged["gpa_numeric"].fillna(merged["gpa_numeric"].median())
merged["tier_score"] = pd.to_numeric(merged["tier_score"], errors="coerce").fillna(50)
merged["min_gpa"] = pd.to_numeric(merged["min_gpa"], errors="coerce").fillna(3.0)
merged["min_gre_quant"] = pd.to_numeric(merged["min_gre_quant"], errors="coerce")

# FIX: filling missing GRE with 0 was actively hurting the model — it looked like a
# catastrophically bad score instead of "no GRE reported", and dragged the GRE column's
# signal toward noise. Impute with the column median instead; gre_missing (flag) already
# tells the model separately whether the value is real or imputed.
merged["gre_numeric"] = merged["gre_numeric"].fillna(merged["gre_numeric"].median())

# NEW: gap/interaction features — these carry much more signal than raw values alone,
# because "3.6 GPA at a program requiring 3.8" matters more than "3.6 GPA" in isolation.
merged["gpa_gap"] = merged["gpa_numeric"] - merged["min_gpa"]
merged["gre_gap"] = merged["gre_numeric"] - merged["min_gre_quant"].fillna(merged["gre_numeric"].median())
merged["gpa_x_tier"] = merged["gpa_numeric"] * (merged["tier_score"] / 100)

# neutral default until Agent 7 provides real scores for this cohort
merged["extracurricular_score"] = 0.5

merged["admitted"] = merged["admit_result"].map({"Admit": 1, "Reject": 0, "Waitlist": 0})

features = [
    "gpa_numeric", "gre_numeric", "gre_missing", "tier_score", "min_gpa",
    "extracurricular_score", "gpa_gap", "gre_gap", "gpa_x_tier"
]
X = merged[features]
y = merged["admitted"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("X_train shape:", X_train.shape, "| X_test shape:", X_test.shape)
class_distribution = y.value_counts(normalize=True)

print("Class distribution:")
print(class_distribution.round(3))

print("Base rate (majority class):", round(class_distribution.max(), 3))

admits.csv columns: ['student_gpa_band', 'student_gre_band', 'program_id', 'admit_result', 'term']
programs.csv columns: ['university_id', 'university_name', 'state', 'city', 'latitude', 'longitude', 'tuition_in_state_usd', 'tuition_out_state_usd', 'university_admission_rate', 'student_size', 'us_news_ranking', 'world_ranking', 'campus_setting', 'website_url', 'avg_cost_of_living_monthly', 'international_student_pct', 'public_or_private', 'program_id', 'program_name', 'degree_type', 'department', 'min_gpa', 'min_gre_quant', 'min_gre_verbal', 'min_gmat', 'min_toefl', 'min_ielts', 'application_deadline_fall', 'application_fee_usd', 'duration_months', 'stem_designated', 'program_admit_rate', 'description_text', 'curriculum_highlights', 'faculty_research_areas', 'avg_starting_salary_usd', 'male_acceptance_rate', 'female_acceptance_rate', 'historical_admit_count', 'historical_admit_rate', 'data_quality', 'track', 'tier_score', 'tier']
X_train shape: (4036, 9) | X_test shape: (1009, 9)
Class

In [13]:
from sklearn.ensemble import RandomForestClassifier


rf = RandomForestClassifier(random_state=42, class_weight="balanced", n_jobs=-1)

param_grid = {
    "n_estimators": [200, 400, 600],
    "max_depth": [5, 10, 15, 20, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}

grid_search = GridSearchCV(
    estimator=rf, param_grid=param_grid, cv=5,
    scoring="accuracy", n_jobs=-1, verbose=1
)
grid_search.fit(X_train, y_train)

admit_model_raw = grid_search.best_estimator_

print("Best Parameters:", grid_search.best_params_)

preds = admit_model_raw.predict(X_test)
print("\nTest Accuracy:", accuracy_score(y_test, preds))
print()
print(classification_report(y_test, preds))
print("Confusion Matrix:")
print(confusion_matrix(y_test, preds))

Fitting 5 folds for each of 270 candidates, totalling 1350 fits
Best Parameters: {'max_depth': 5, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 10, 'n_estimators': 200}

Test Accuracy: 0.5708622398414271

              precision    recall  f1-score   support

           0       0.53      0.58      0.56       466
           1       0.61      0.56      0.58       543

    accuracy                           0.57      1009
   macro avg       0.57      0.57      0.57      1009
weighted avg       0.57      0.57      0.57      1009

Confusion Matrix:
[[271 195]
 [238 305]]


### 6a. Honest note on the 85% accuracy target
With only band-level GPA/GRE (`3.4-3.5` style buckets, not exact values) and a handful of
program-level columns, real admissions decisions have a lot of signal this dataset simply
doesn't capture — SOP quality, recommendation strength, interview performance, applicant pool
depth that year, etc. **56% accuracy on a roughly 54%-majority-class problem means the model
was barely better than guessing the majority class** — that's a feature-signal problem, not a
hyperparameter problem, and no amount of grid-searching the same 5 weak features will reach 85%.

What actually moves the needle, in priority order:
1. **Gap/interaction features** (added below: `gpa_gap`, `gre_gap`, `gpa_x_tier`) — almost always
   the single biggest lift for admit-probability problems, since "how you compare to the bar"
   matters more than your raw score alone.
2. **More columns from `admits.csv`** — check the printed column list in the cell above. If it has
   undergrad institution tier, work-experience years, major, or an SOP/LOR score (once Agent 7
   exists), add them — this is the highest-leverage fix available.
3. **Model comparison** — RandomForest isn't always best on tabular data this size; the cell below
   also tries GradientBoosting and HistGradientBoosting and picks the strongest by CV.
4. **Recalibrate expectations** — if after (1)-(3) accuracy plateaus in the 60s-70s, that may be the
   honest ceiling for this data, and forcing 85% would mean overfitting to noise. Report the
   Brier score and calibration curve (Section 6b) as the more meaningful measure of trustworthiness
   for something you're calling an "admission probability."

In [15]:
from sklearn.ensemble import GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_score

rf = RandomForestClassifier(random_state=42, class_weight="balanced", n_jobs=-1)

param_grid = {
    "n_estimators": [200, 400, 600],
    "max_depth": [5, 10, 15, 20, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}

grid_search = GridSearchCV(
    estimator=rf, param_grid=param_grid, cv=5,
    scoring="accuracy", n_jobs=-1, verbose=1
)
grid_search.fit(X_train, y_train)
rf_best = grid_search.best_estimator_
print("RF best params:", grid_search.best_params_)

gb = GradientBoostingClassifier(random_state=42)
gb.fit(X_train, y_train)

hgb = HistGradientBoostingClassifier(random_state=42, class_weight="balanced")
hgb.fit(X_train, y_train)

candidates = {"RandomForest (tuned)": rf_best, "GradientBoosting": gb, "HistGradientBoosting": hgb}
results = {}
for name, clf in candidates.items():
    cv_acc = cross_val_score(clf, X_train, y_train, cv=5, scoring="accuracy").mean()
    test_acc = accuracy_score(y_test, clf.predict(X_test))
    results[name] = (cv_acc, test_acc)
    print(f"{name}: CV accuracy={cv_acc:.3f} | test accuracy={test_acc:.3f}")

best_name = max(results, key=lambda k: results[k][0])
admit_model_raw = candidates[best_name]
print(f"\nSelected model: {best_name}")

preds = admit_model_raw.predict(X_test)
print("\nTest Accuracy:", accuracy_score(y_test, preds))
print()
print(classification_report(y_test, preds))
print("Confusion Matrix:")
print(confusion_matrix(y_test, preds))
print()
print("NOTE: if this is still well under 85%, re-read the markdown note above —")
print("the fix at this point is richer features (esp. from admits.csv columns), not more tuning.")

Fitting 5 folds for each of 270 candidates, totalling 1350 fits
RF best params: {'max_depth': 5, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 10, 'n_estimators': 200}
RandomForest (tuned): CV accuracy=0.606 | test accuracy=0.571
GradientBoosting: CV accuracy=0.603 | test accuracy=0.575
HistGradientBoosting: CV accuracy=0.570 | test accuracy=0.560

Selected model: RandomForest (tuned)

Test Accuracy: 0.5708622398414271

              precision    recall  f1-score   support

           0       0.53      0.58      0.56       466
           1       0.61      0.56      0.58       543

    accuracy                           0.57      1009
   macro avg       0.57      0.57      0.57      1009
weighted avg       0.57      0.57      0.57      1009

Confusion Matrix:
[[271 195]
 [238 305]]

NOTE: if this is still well under 85%, re-read the markdown note above —
the fix at this point is richer features (esp. from admits.csv columns), not more tuning.


In [16]:
from sklearn.inspection import permutation_importance

# Works regardless of which model type was selected above (RF / GB / HistGB) —
# unlike .feature_importances_, which HistGradientBoostingClassifier doesn't expose.
perm = permutation_importance(admit_model_raw, X_test, y_test, n_repeats=20, random_state=42, n_jobs=-1)

importance_df = pd.DataFrame({
    "feature": features,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values("importance_mean", ascending=False)

print(importance_df)

                 feature  importance_mean  importance_std
0            gpa_numeric         0.014470        0.011245
6                gpa_gap         0.007185        0.007605
4                min_gpa         0.004708        0.002123
8             gpa_x_tier         0.002230        0.009780
2            gre_missing         0.000099        0.001595
5  extracurricular_score         0.000000        0.000000
7                gre_gap        -0.000248        0.009770
1            gre_numeric        -0.001487        0.006510
3             tier_score        -0.002527        0.002881


In [17]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import brier_score_loss

# Calibrate the selected raw admission model
admit_model = CalibratedClassifierCV(
    estimator=admit_model_raw,
    method="sigmoid",
    cv=5
)

admit_model.fit(X_train, y_train)

# Evaluate calibrated probabilities on untouched test data
positive_idx = list(admit_model.classes_).index(1)
test_prob = admit_model.predict_proba(X_test)[:, positive_idx]

print("Calibrated model:", type(admit_model).__name__)
print("Test Brier Score:", round(brier_score_loss(y_test, test_prob), 4))

Calibrated model: CalibratedClassifierCV
Test Brier Score: 0.2386


## 7. SHAP explainability
SHAP explains the underlying `admit_model_raw` (the tuned RandomForest) — `CalibratedClassifierCV` wraps it for probability output, but SHAP's `TreeExplainer` needs the tree model directly. Positive-class index is detected at runtime rather than assumed.

In [26]:
import shap

shap_explainer = shap.TreeExplainer(admit_model_raw)

# Run once on a sample to detect this SHAP version's output shape/format
sample_shap = shap_explainer.shap_values(X_test.iloc[[0]])
print("Raw shap_values type:", type(sample_shap))
if isinstance(sample_shap, list):
    print("List of", len(sample_shap), "arrays (one per class) — shape per class:", sample_shap[0].shape)
    POSITIVE_CLASS_INDEX = list(admit_model_raw.classes_).index(1)
    print("Detected positive class (admit=1) at index:", POSITIVE_CLASS_INDEX)
else:
    print("Single array shape:", sample_shap.shape)
    POSITIVE_CLASS_INDEX = None  # handled directly in get_shap_top_factors


def get_shap_top_factors(X_row_df, top_n=3):
    """
    Extract the top SHAP factors for the positive/admit class.
    Disables SHAP's additivity check because small numerical
    differences can occur between the trained tree model and
    the SHAP explainer.
    """

    shap_values = shap_explainer.shap_values(
        X_row_df,
        check_additivity=False
    )

    # Handle older SHAP: list of arrays, one array per class
    if isinstance(shap_values, list):
        positive_idx = list(admit_model.classes_).index(1)
        values = shap_values[positive_idx][0]

    # Handle newer SHAP: numpy array
    else:
        values = shap_values[0]

        # If output is 3D: (samples, features, classes)
        if len(values.shape) == 2 and values.shape[1] == len(admit_model.classes_):
            positive_idx = list(admit_model.classes_).index(1)
            values = values[:, positive_idx]

    feature_names = X_row_df.columns.tolist()

    factor_pairs = list(zip(feature_names, values))

    # Sort by absolute SHAP impact
    factor_pairs = sorted(
        factor_pairs,
        key=lambda x: abs(float(x[1])),
        reverse=True
    )

    top_factors = []

    for feature, value in factor_pairs[:top_n]:
        direction = "increases" if value > 0 else "decreases"

        top_factors.append({
            "feature": feature,
            "impact": round(float(value), 4),
            "direction": direction
        })

    return top_factors

Raw shap_values type: <class 'numpy.ndarray'>
Single array shape: (1, 9, 2)


In [19]:
print("admit_model:", type(admit_model))
print("admit_model_raw:", type(admit_model_raw))
print("shap_explainer:", type(shap_explainer))

admit_model: <class 'sklearn.calibration.CalibratedClassifierCV'>
admit_model_raw: <class 'sklearn.ensemble._forest.RandomForestClassifier'>
shap_explainer: <class 'shap.explainers._tree.TreeExplainer'>


In [20]:
import joblib

joblib.dump(admit_model, "agent2_admit_predictor_calibrated.pkl")
joblib.dump(admit_model_raw, "agent2_admit_predictor_raw.pkl")
joblib.dump(shap_explainer, "agent2_shap_explainer.pkl")

from google.colab import files
files.download("agent2_admit_predictor_calibrated.pkl")
files.download("agent2_admit_predictor_raw.pkl")
files.download("agent2_shap_explainer.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 8. Tier + admit probability at inference time
Uses the **calibrated** model for `predict_proba` (real deployment probability) and the **raw** model for SHAP (tree-based explainability). Positive-class index is looked up via `.classes_` instead of assuming `[0][1]`. GPA is normalized before use, matching training.

In [29]:
def compute_tier_ml(profile, payload, extracurricular_score):
    # Normalize GPA to 4.0 scale
    normalized_gpa = normalize_gpa_to_4(
        profile.get("gpa"),
        profile.get("gpa_scale")
    )

    student_gpa = float(normalized_gpa or 3.0)

    # GRE
    gre_value = profile.get("gre_quant")
    if gre_value is None or gre_value == "":
        student_gre = 0.0
        gre_missing = 1
    else:
        student_gre = float(gre_value)
        gre_missing = 0

    # Program requirements
    tier_score = float(payload.get("tier_score", 50) or 50)
    min_gpa = float(payload.get("min_gpa", 3.0) or 3.0)

    min_gre_quant = payload.get("min_gre_quant")
    if min_gre_quant is None or min_gre_quant == "":
        min_gre_quant = None
    else:
        min_gre_quant = float(min_gre_quant)

    # ---- SAME DERIVED FEATURES USED DURING TRAINING ----

    gpa_gap = student_gpa - min_gpa

    if min_gre_quant is not None:
        gre_gap = student_gre - min_gre_quant
    else:
        gre_gap = 0.0

    gpa_x_tier = student_gpa * (tier_score / 100)

    # ---- BUILD ALL 9 FEATURES ----

    X_new = pd.DataFrame(
        [[
            student_gpa,
            student_gre,
            gre_missing,
            tier_score,
            min_gpa,
            float(extracurricular_score),
            gpa_gap,
            gre_gap,
            gpa_x_tier
        ]],
        columns=features
    )

    # Admission probability
    positive_idx = list(admit_model.classes_).index(1)
    admit_prob = float(
        admit_model.predict_proba(X_new)[0][positive_idx]
    )

    # SHAP explanation
    shap_values = shap_explainer.shap_values(
      X_new,
      check_additivity=False
    )

    # Version-safe positive-class SHAP extraction
    if isinstance(shap_values, list):
        positive_shap = shap_values[positive_idx][0]
    else:
        positive_shap = shap_values[0]

    top_factors = get_shap_top_factors(X_new)

    # Admission tier
    if admit_prob >= 0.65:
        tier = "Safe"
    elif admit_prob >= 0.35:
        tier = "Target"
    else:
        tier = "Ambitious"

    return tier, admit_prob, top_factors, student_gpa

## 9. Explanation generation (Groq LLaMA — grounded in SHAP factors + normalized GPA)

In [32]:
from groq import Groq

groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])


def generate_explanation(university_name, program_name, normalized_gpa, target_program, tier, score, top_factors, description_text=""):
    factors_str = ", ".join(f"{f['feature']} ({'+' if f['impact']>=0 else ''}{f['impact']})" for f in top_factors)
    prompt = (
        f"Explain in 2 sentences why {university_name}'s {program_name} "
        f"is a {tier} match for a student with GPA {normalized_gpa:.2f}/4.0 targeting {target_program}. "
        f"Score: {score}. Ground your explanation in these weighted factors from the model: {factors_str}. "
        f"Mention one concrete detail: {description_text[:300]}"
    )

    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-120b",
        max_tokens=400,
        reasoning_effort="low",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

## 10. Persistence — save matches to Postgres

In [23]:
from sqlalchemy import create_engine, text
import json as _json

def save_matches_to_db(student_id, university_matches):
    if not POSTGRES_URL:
        print("POSTGRES_URL not set — skipping DB persistence (in-memory only).")
        return
    engine = create_engine(POSTGRES_URL)
    with engine.begin() as conn:
        conn.execute(text("""
            CREATE TABLE IF NOT EXISTS university_matches (
                student_id TEXT,
                university TEXT,
                country TEXT,
                region TEXT,
                type TEXT,
                admit_probability FLOAT,
                overall_score FLOAT,
                reasoning TEXT,
                programs JSONB,
                created_at TIMESTAMP DEFAULT now()
            )
        """))
        for m in university_matches:
            conn.execute(text("""
                INSERT INTO university_matches
                (student_id, university, country, region, type, admit_probability, overall_score, reasoning, programs)
                VALUES (:sid, :uni, :country, :region, :type, :prob, :overall, :reasoning, :programs)
            """), {
                "sid": student_id, "uni": m["university"], "country": m["country"],
                "region": m["region"], "type": m["type"], "prob": m["admit_probability"],
                "overall": m["overall_score"], "reasoning": m["reasoning"], "programs": _json.dumps(m["programs"])
            })
    print(f"Saved {len(university_matches)} university matches for student {student_id} to Postgres.")

## 11. Full agent function
University ranking now uses a combined `overall_score` (mean program `match_score` at that university + best `admit_probability`, weighted), not admit_probability alone — a university that's a much better semantic/profile fit can outrank one with a marginally higher admit probability. Program-level detail stays nested under `programs` for Agent 3.

In [33]:
def university_matching_agent(state):
    profile = state["profile"]
    preferences = state.get("preferences", {})
    priority = preferences.get("priority", "default")
    extracurricular_score = state.get("extracurricular", {}).get("profile_strength_score", 0.5)

    candidates = retrieve_candidates(profile, preferences, top_k=20)
    query_text = f"{profile['target_degree']} {profile['target_program']}"
    reranked = rerank(query_text, candidates, top_n=10)

    # Score each program-level candidate first (retains program granularity internally)
    program_level_matches = []
    for candidate, sem_score in reranked:
        payload = candidate.payload
        score = compute_final_score(payload, profile, sem_score, extracurricular_score, priority)
        tier, admit_prob, top_factors, normalized_gpa = compute_tier_ml(profile, payload, extracurricular_score)
        explanation = generate_explanation(
            payload["university_name"], payload["program_name"], normalized_gpa,
            profile["target_program"], tier, score, top_factors, payload.get("description_text", "")
        )
        program_level_matches.append({
            "university_name": payload["university_name"],
            "program_name": payload["program_name"],
            "country": payload.get("country", "Unknown"),
            "region": payload.get("region", "Unknown"),
            "type": payload.get("institution_type", "Unknown"),
            "tier": tier,
            "match_score": score,
            "admit_probability": round(float(admit_prob), 3),
            "within_stated_budget": tag_budget_signal(payload, preferences),
            "explainability": top_factors,
            "explanation": explanation
        })

    # Group into university-level shortlist (framework's expected output shape)
    by_university = {}
    for m in program_level_matches:
        key = m["university_name"]
        if key not in by_university:
            by_university[key] = {
                "university": m["university_name"],
                "country": m["country"],
                "region": m["region"],
                "type": m["type"],
                "programs": [],
            }
        by_university[key]["programs"].append(m)

    university_matches = []
    for uni in by_university.values():
        programs_here = uni["programs"]
        best_program = max(programs_here, key=lambda p: p["admit_probability"])
        avg_match_score = sum(p["match_score"] for p in programs_here) / len(programs_here)

        uni["admit_probability"] = best_program["admit_probability"]
        # Combined ranking signal: university match quality + admission likelihood,
        # so a strong semantic/profile fit isn't buried under a merely higher admit_probability.
        uni["overall_score"] = round(0.6 * avg_match_score + 0.4 * best_program["admit_probability"], 3)
        uni["reasoning"] = best_program["explanation"]
        university_matches.append(uni)

    university_matches = sorted(university_matches, key=lambda u: u["overall_score"], reverse=True)

    state["matched_universities"] = university_matches
    state["status"] = "university_done"

    save_matches_to_db(state.get("student_id", "unknown"), university_matches)
    return state

## 12. Test run

In [34]:
MOCK_STATE = {
    "student_id": "test-001",
    "profile": {
        "gpa": 8.6, "gpa_scale": 10.0,   # deliberately non-4.0 scale to verify the GPA fix
        "target_program": "Computer Science", "target_degree": "MS",
        "test_scores": {"GRE": {"quant": 165}}
    },
    "preferences": {"budget_max_usd": 45000, "priority": "research"},
    "extracurricular": {"profile_strength_score": 0.72}
}

result = university_matching_agent(MOCK_STATE)
for u in result["matched_universities"]:
    print(f"{u['university']} ({u['type']}, {u['region']}, {u['country']}) — overall_score={u['overall_score']} admit_probability={u['admit_probability']}")
    print(f"  reasoning: {u['reasoning']}")
    for p in u["programs"]:
        print(f"    - {p['program_name']} | tier={p['tier']} match_score={p['match_score']} within_budget={p['within_stated_budget']}")
    print()

POSTGRES_URL not set — skipping DB persistence (in-memory only).
Stanford University (Unknown, Unknown, Unknown) — overall_score=0.699999988079071 admit_probability=0.566
  reasoning: Stanford’s MS Computer Science is a Target match for a student with a 3.44 / 4.0 GPA because the model’s positive gpa_x_tier (+0.0461) and tier_score (+0.0326) outweigh the modest negative impact of the GRE‑gap (‑0.0241), resulting in a strong overall fit score of 0.787. Moreover, the program—an advanced graduate degree in Computer & Information Sciences—focuses on Artificial Intelligence, Systems, and Algorithms and lets students deepen their expertise through electives and project‑based work.
    - MS Computer Science | tier=Target match_score=0.7870000004768372 within_budget=False
    - MS Computer Science - Systems | tier=Target match_score=0.7929999828338623 within_budget=False

University of California-Berkeley (Unknown, Unknown, Unknown) — overall_score=0.6729999780654907 admit_probability=0.51
  r

In [31]:
print("embed_model type:", type(embed_model))
print("admit_model (calibrated) type:", type(admit_model))
print("admit_model_raw type:", type(admit_model_raw))
print("shap_explainer type:", type(shap_explainer))

embed_model type: <class 'sentence_transformers.sentence_transformer.model.SentenceTransformer'>
admit_model (calibrated) type: <class 'sklearn.calibration.CalibratedClassifierCV'>
admit_model_raw type: <class 'sklearn.ensemble._forest.RandomForestClassifier'>
shap_explainer type: <class 'shap.explainers._tree.TreeExplainer'>
